In [1]:
import ollama
import json
import pandas as pd
import re
from tqdm import tqdm
import numpy as np
import warnings
from sklearn.model_selection import train_test_split  # 데이터 분할을 위한 라이브러리 추가

# --- 설정 변수 ---
file_path = './Data/CorrectAnswer/'
file_name = 'CorrectAnswer_v3.csv'

model_name = 'gemma3:4b'
model_save_name = 'Gemma3_4b'

# --- RAG 라이브러리 로드 ---
try:
    import faiss
    from sentence_transformers import SentenceTransformer
    print("RAG 라이브러리 (faiss, sentence-transformers, sklearn) 로드 완료.")
except ImportError:
    print("[오류] 필수 라이브러리가 없습니다. 'pip install sentence-transformers faiss-cpu scikit-learn'을 실행하세요.")

warnings.filterwarnings('ignore')

/opt/anaconda3/envs/llmenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG 라이브러리 (faiss, sentence-transformers, sklearn) 로드 완료.


In [3]:
k_value = 3
train_test_rate = 0.1

In [5]:
# 1. 지식 베이스(Vector DB) 구축 함수 - (수정됨: DataFrame을 직접 받음)
def build_vector_db_from_df(df_source, model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
    """
    DataFrame을 입력받아 FAISS Vector DB를 구축합니다.
    이 데이터는 RAG의 '참고서(Reference)' 역할을 하는 훈련 데이터(90%)입니다.
    """
    print(f"지식 베이스(DB) 구축 시작... (데이터 개수: {len(df_source)}개)")

    # 임베딩 모델 로드
    try:
        embed_model = SentenceTransformer(model_name)
    except Exception as e:
        print(f"  [치명적 오류] 임베딩 모델 로드 실패: {e}")
        return None, None
    
    # 'Original_Name'을 임베딩 (결측치 처리 포함)
    knowledge_names = df_source['Original_Name'].fillna('').astype(str).tolist()
    
    try:
        # 임베딩 수행
        knowledge_vectors = embed_model.encode(knowledge_names, show_progress_bar=True)
    except Exception as e:
        print(f"  [치명적 오류] 임베딩 중 오류 발생: {e}")
        return None, None
    
    # FAISS 인덱스 구축
    dimension = knowledge_vectors.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(knowledge_vectors.astype('float32'))
    
    print(f"FAISS Vector DB 구축 완료 (d={dimension}).")
    return index, embed_model

In [7]:
# 2. RAG 프롬프트 템플릿 정의
PROMPT_TEMPLATE_RAG = """
당신은 한국어 하드웨어 공구 이름 분류 전문가입니다. 당신의 임무는 정제되어있지 않은 공구 이름을 분석하여 구조화된 정보를 추출하는 것입니다.

JSON 스키마는 다음과 같습니다:
{{
  "Preprocessed_Name": "공구의 표준화된 기본 유형입니다. 한국산업 표준 용어로 '반드시' 표준화하세요 (예: 기리 -> 드릴비트, 셋트 -> 세트). 띄어쓰기를 하지 마세요.",
  "Brand": "제조사 또는 브랜드 이름입니다. 없는 경우 null을 사용하세요.",
  "Power_Source": "전원 공급 방식입니다 (예: 무선, 유선, 수동). 없는 경우 null을 사용하세요.",
  "Specification": "기타 사양입니다. 없는 경우 null을 사용하세요."
}}

---
[중요] 아래는 당신이 참고해야 할, 입력 데이터와 가장 유사한 **"정답 데이터베이스"**의 검색 결과입니다.
이 정보를 **최우선으로 참고**하여 JSON을 생성하세요.

{context}
---

이제, 위의 "정답 데이터베이스" 내용을 바탕으로 다음 공구 이름을 분석하고 JSON 출력을 제공하세요.

Input Name: '{tool_name}'
Output:
"""

In [9]:
# 3. RAG 워크플로우 실행 함수
def run_rag_workflow(tool_name, index, embed_model, df_knowledge, k=3):
    """
    tool_name: 정제할 대상 (Test Data)
    index: 훈련 데이터로 만든 Vector DB
    df_knowledge: 훈련 데이터 DataFrame (상세 정보 참조용)
    """
    
    # 1. Retrieve (검색)
    query_vector = embed_model.encode([str(tool_name).strip()]).astype('float32')
    distances, indices = index.search(query_vector, k)
    
    # 검색된 인덱스에 해당하는 훈련 데이터 정보를 가져옴
    retrieved_docs = df_knowledge.iloc[indices[0]]
    
    # 2. Augment (프롬프트 주입)
    context_str = "--- (검색된 유사 사례) ---\n"
    for i, (idx, row) in enumerate(retrieved_docs.iterrows()):
        context_str += f"[유사 사례 {i+1}] (입력: '{row['Original_Name']}')\n"
        context_str += f"  > 정답 (JSON): {{"
        context_str += f"\"Preprocessed_Name\": \"{row['Tool_Type']}\""
        if pd.notna(row['Brand']):
            context_str += f", \"Brand\": \"{row['Brand']}\""
        if pd.notna(row['Power_Source']):
             context_str += f", \"Power_Source\": \"{row['Power_Source']}\""
        context_str += f"}}\n"
    context_str += "--- (유사 사례 끝) ---"
    
    formatted_prompt = PROMPT_TEMPLATE_RAG.format(
        tool_name=str(tool_name).strip(),
        context=context_str
    )
    
    # 3. Generate (생성)
    try:
        response = ollama.generate(
            model=model_name,
            prompt=formatted_prompt,
            format='json',
            stream=False,
            options={"temperature": 0}
        )
        response_text = response.get('response', '{}')
        json_output = json.loads(response_text)
        return {'status': 'success', 'processed_data': json_output}
    
    except Exception as e:
        # print(f"  [LLM 오류] {tool_name}: {e}")
        return {'status': 'error', 'message': str(e)}

In [11]:
# --- 메인 실행 로직 ---

# 1. 전체 데이터 로드
print("전체 데이터를 로드합니다...")
try:
    df_total = pd.read_csv(file_path + file_name)
    if 'Original_Name' not in df_total.columns:
        raise ValueError("'Original_Name' 컬럼이 필요합니다.")
except Exception as e:
    print(f"데이터 로드 실패: {e}")
    exit()

전체 데이터를 로드합니다...


In [13]:
# 2. 데이터 분할 (Train 70% : Test 30%)
# random_state를 고정하여 매번 실행 시 같은 데이터가 분할되도록 함
print("\n[데이터 분할] 훈련용(DB) 90% vs 테스트용(입력) 10% 로 분리합니다...")
df_train, df_test = train_test_split(df_total, test_size=train_test_rate, random_state=42)

# 인덱스 리셋 (iloc 접근을 위해 필수)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f" - 전체 데이터 수: {len(df_total)}")
print(f" - 훈련 데이터(DB 구축용): {len(df_train)}")
print(f" - 테스트 데이터(정제 대상): {len(df_test)}")


[데이터 분할] 훈련용(DB) 90% vs 테스트용(입력) 10% 로 분리합니다...
 - 전체 데이터 수: 3352
 - 훈련 데이터(DB 구축용): 3016
 - 테스트 데이터(정제 대상): 336


In [15]:
# 3. 지식 베이스(Vector DB) 구축 - *훈련 데이터(df_train)만 사용*
faiss_index, embed_model = build_vector_db_from_df(df_train)

processed_data_list_rag = []

if faiss_index is not None:
    print(f"\n--- RAG 정제 시작 (테스트 데이터 {len(df_test)}건 처리) ---")
    
    # 4. 테스트 데이터에 대해 RAG 실행
    # df_test의 Original_Name을 하나씩 꺼내서 입력으로 사용
    test_tools = df_test['Original_Name'].tolist()
    
    for tool in tqdm(test_tools, desc="Test Data 정제 중"):
        # *중요*: 여기서 df_knowledge 인자에는 'df_train'을 넣어줍니다.
        # (테스트 데이터에 대한 답을 찾기 위해 훈련 데이터 DB를 참조)
        result = run_rag_workflow(tool, faiss_index, embed_model, df_train, k=3)
        
        result_entry = {
            'Input_Name': tool, # 입력값 (Original Name)
            'status': result['status']
        }
        
        if result['status'] == 'success':
            result_entry.update(result['processed_data'])
        else:
            result_entry['error_message'] = result.get('message')
            
        processed_data_list_rag.append(result_entry)
        
    print("RAG 정제 및 평가 완료.")
else:
    print("[실행 중단] Vector DB 구축 실패.")

지식 베이스(DB) 구축 시작... (데이터 개수: 3016개)


Batches: 100%|██████████████████████████████████| 95/95 [00:07<00:00, 13.56it/s]


FAISS Vector DB 구축 완료 (d=384).

--- RAG 정제 시작 (테스트 데이터 336건 처리) ---


Test Data 정제 중: 100%|██████████████████████| 336/336 [43:18<00:00,  7.73s/it]

RAG 정제 및 평가 완료.


In [16]:
# %% [markdown]
# ### 4. 결과 저장 및 확인

# %%
if processed_data_list_rag:
    df_rag_results = pd.DataFrame(processed_data_list_rag)
    
    # 파일명에 'TrainTestSplit' 표시 추가
    save_path = file_path + 'New_RAG_' + model_save_name + '_0.1.csv'
    
    
    # 띄어쓰기 제거 (후처리)
    if 'Preprocessed_Name' in df_rag_results.columns:
        df_rag_results['Preprocessed_Name'] = df_rag_results['Preprocessed_Name'].str.replace(' ','')
        df_rag_results.to_csv(save_path, index=False)
    print(f"\n--- 결과가 저장되었습니다: {save_path} ---")
    print("상위 5개 결과 미리보기:")
    try:
        display(df_rag_results.head())
    except NameError:
        print(df_rag_results.head())
else:
    print("처리된 결과가 없습니다.")


--- 결과가 저장되었습니다: ./Data/CorrectAnswer/New_RAG_Gemma3_4b_0.1.csv ---
상위 5개 결과 미리보기:


,Input_Name,status,Preprocessed_Name,Brand,Power_Source,Specification
0,톱(대),success,톱,NaN,NaN,NaN
1,방역기,success,방역기,NaN,NaN,NaN
2,차단릴,success,릴,NaN,NaN,NaN
3,몽키 10,success,몽키스패너,NaN,NaN,NaN
4,톱(소형),success,톱,NaN,NaN,NaN


In [17]:
df_rag_results

,Input_Name,status,Preprocessed_Name,Brand,Power_Source,Specification
0,톱(대),success,톱,NaN,NaN,NaN
1,방역기,success,방역기,NaN,NaN,NaN
2,차단릴,success,릴,NaN,NaN,NaN
3,몽키 10,success,몽키스패너,NaN,NaN,NaN
4,톱(소형),success,톱,NaN,NaN,NaN
...,...,...,...,...,...,...
331,쇠톱(1),success,톱,NaN,NaN,NaN
332,블루투스 스피커,success,스피커,NaN,NaN,NaN
333,톱 1개,success,톱,NaN,NaN,NaN
334,5단 사다리,success,사다리,NaN,NaN,NaN
